# 04 — Prepare YOLO Detection Dataset

**Purpose:** Convert the annotated dataset into YOLO format for training the region detector.

The detector learns from full document images with bounding box labels.
YOLO requires a specific folder layout and normalized coordinates.

This notebook:
1. Converts bbox annotations to YOLO normalized format.
2. Validates every label file.
3. Draws YOLO labels back onto images for visual verification.
4. Confirms `data/yolo/data.yaml` is correct.

In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f'Project root: {PROJECT_ROOT}')

In [ ]:
# ── Verify prerequisites ──────────────────────────────────────────
from src.utils.paths import get_path

prereqs = [
    ('train_split', 'data/processed/train_split.jsonl'),
    ('val_split',   'data/processed/val_split.jsonl'),
]
all_ok = True
for key, label in prereqs:
    p = get_path(key)
    ok = p.exists()
    icon = '✅' if ok else '❌'
    print(f'{icon} {label}')
    if not ok:
        all_ok = False

if not all_ok:
    print('\nRun these first:')
    print('  python -m src.data.make_splits')

In [ ]:
# ── Convert to YOLO format ────────────────────────────────────────
# This script:
#   1. Reads train_split.jsonl and val_split.jsonl
#   2. For EVERY region (all 7 types including image & graph),
#      writes a YOLO label row: class_id xc yc w h (normalized 0-1)
#   3. Copies images to data/yolo/images/train/ and images/val/
#   4. Writes data/yolo/data.yaml

from src.utils.jsonl import read_jsonl
from src.utils.paths import get_class_to_id
from src.detection.convert_to_yolo import convert_records, write_yolo_data_yaml

class_map   = get_class_to_id()
images_root = get_path('raw_train_images')

for split_name, jsonl_key, img_key, lbl_key in [
    ('train', 'train_split', 'yolo_train_images', 'yolo_train_labels'),
    ('val',   'val_split',   'yolo_val_images',   'yolo_val_labels'),
]:
    jsonl_path = get_path(jsonl_key)
    if not jsonl_path.exists():
        print(f'⚠️  {jsonl_path} not found — skipping {split_name}')
        continue

    records = read_jsonl(jsonl_path)
    n_img, n_reg, n_skip = convert_records(
        records,
        images_src  = images_root,
        out_img_dir = get_path(img_key),
        out_lbl_dir = get_path(lbl_key),
        class_map   = class_map,
        copy_images = True,
    )
    print(f'  {split_name}: {n_img:,} images, {n_reg:,} regions, {n_skip:,} skipped')

write_yolo_data_yaml(
    train_img_dir = get_path('yolo_train_images'),
    val_img_dir   = get_path('yolo_val_images'),
    class_map     = class_map,
    out_path      = get_path('yolo_data_yaml'),
)
print('\n✅ YOLO dataset prepared.')

In [ ]:
# ── Validate YOLO dataset ─────────────────────────────────────────

from src.detection.validate_yolo import (
    validate_split, validate_data_yaml, print_summary, save_report
)

yaml_errs = validate_data_yaml(get_path('yolo_data_yaml'))

train_s = validate_split(
    get_path('yolo_train_images'), get_path('yolo_train_labels'), 'train'
)
val_s = validate_split(
    get_path('yolo_val_images'), get_path('yolo_val_labels'), 'val'
)

print_summary(train_s, val_s, yaml_errs)
save_report(train_s, val_s, yaml_errs)

In [ ]:
# ── Save YOLO visual debug images ────────────────────────────────
# Draws YOLO label boxes back onto images to confirm coordinates are correct.

from src.detection.validate_yolo import save_yolo_debug_images
from src.utils.paths import ensure_dir

for split in ['train', 'val']:
    img_dir   = get_path(f'yolo_{split}_images')
    lbl_dir   = get_path(f'yolo_{split}_labels')
    debug_dir = ensure_dir(get_path('debug_images_dir') / 'yolo_checks' / split)
    if img_dir.exists():
        save_yolo_debug_images(img_dir, lbl_dir, debug_dir, n=6)

print('\n✅ YOLO debug images saved to outputs/debug_images/yolo_checks/')

In [ ]:
# ── Display YOLO debug images ─────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

debug_dir = get_path('debug_images_dir') / 'yolo_checks' / 'train'
debug_imgs = sorted(debug_dir.glob('yolo_debug_*.jpg'))[:4] if debug_dir.exists() else []

if debug_imgs:
    fig, axes = plt.subplots(1, len(debug_imgs), figsize=(18, 6))
    if len(debug_imgs) == 1:
        axes = [axes]
    for ax, p in zip(axes, debug_imgs):
        ax.imshow(mpimg.imread(str(p)))
        ax.set_title(p.name[:30], fontsize=7)
        ax.axis('off')
    plt.suptitle('YOLO Debug — boxes must align with actual regions', fontsize=11)
    plt.tight_layout()
    plt.show()
else:
    print('No YOLO debug images found — check that images were copied correctly.')

In [ ]:
# ── Show data.yaml content ────────────────────────────────────────
yaml_path = get_path('yolo_data_yaml')
if yaml_path.exists():
    print(yaml_path.read_text(encoding='utf-8'))
else:
    print('data.yaml not found.')

## ✅ Next Step

```bash
# Run the final readiness gate:
python -m src.data.foundation_readiness

# Then open:
# notebooks/05_local_validation.ipynb
```